## Google Colab

| Component | Spec |
|---|---|
| GPU | NVIDIA Tesla T4 |
| GPU Memory | 16 GB GDDR6 |
| GPU Architecture | Turing |
| CUDA Cores | 2,560 |
| Tensor Cores | 320 (Gen 2) |
| FP32 Performance | ~8.1 TFLOPS |
| FP16 Performance | ~65 TFLOPS (Tensor) |
| INT8 Performance | ~130 TOPS |
| vCPUs | 2 |
| System RAM | ~12.7 GB (upgradeable to ~25.5 GB via "High-RAM" runtime, Pro only) |
| Disk | ~78–107 GB (varies, ephemeral) |
| CUDA Version | Depends on preinstalled driver (usually CUDA 12.x as of 2026) |



## 1 · Install dependencies

In [ ]:
import subprocess, sys

pkgs = [
    "transformers", "datasets", "scikit-learn",
    "pandas", "numpy", "torch", "torchao",
    "bitsandbytes", "optimum", "quanto", "peft", "accelerate"
]
for p in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])
print("All packages ready.")


In [ ]:
!pip install -U torchao

## 2 · Imports & seeds

In [ ]:
import os, gc, time, warnings, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")


In [ ]:
import subprocess

def detect_gpu():
    if not torch.cuda.is_available():
        return "cpu", 0
    name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    return name, vram_gb

GPU_NAME, GPU_VRAM_GB = detect_gpu()
print(f"GPU detected : {GPU_NAME}")
print(f"VRAM         : {GPU_VRAM_GB:.1f} GB")

# Classify tier
if GPU_VRAM_GB >= 70:
    GPU_TIER = "high"
elif GPU_VRAM_GB >= 40:
    GPU_TIER = "mid"
else:
    GPU_TIER = "low"

print(f"GPU tier     : {GPU_TIER}")


## 3 · Config

In [ ]:
import os

MODEL_NAME = "google/flan-t5-xl"   # small | base | large | xl — change and re-run from here down

T4_PROFILES = {
    "flan-t5-small": dict(batch=16, grad_accum=2, max_in=256, max_out=16,
                          calib=256, train_n=20000, recovery_n=3000, test_batch=32, use_lora=False),
    "flan-t5-base":  dict(batch=8,  grad_accum=4, max_in=192, max_out=16,
                          calib=192, train_n=8000,  recovery_n=1500, test_batch=16, use_lora=False),
    "flan-t5-large": dict(batch=2,  grad_accum=16, max_in=128, max_out=8,
                          calib=96,  train_n=2000,  recovery_n=500,  test_batch=8,  use_lora=True),
    "flan-t5-xl":    dict(batch=1,  grad_accum=32, max_in=96,  max_out=8,
                          calib=64,  train_n=500,   recovery_n=200,  test_batch=4,  use_lora=True),
}

short_name = MODEL_NAME.split("/")[-1]
profile    = T4_PROFILES.get(short_name, T4_PROFILES["flan-t5-base"])

BATCH_SIZE      = profile["batch"]
GRAD_ACCUM      = profile["grad_accum"]
MAX_INPUT_LEN   = profile["max_in"]
MAX_TARGET_LEN  = profile["max_out"]
CALIB_SIZE      = profile["calib"]
BASE_TRAIN_SIZE = profile["train_n"]
RECOVERY_SIZE   = profile["recovery_n"]
TEST_BATCH      = profile["test_batch"]
USE_LORA        = profile["use_lora"]

FINETUNE_EPOCHS = 1
RECOVERY_EPOCHS = 1
CLIP_NORM       = 1.0
LR              = 5e-5
LR_RECOVERY     = 1e-4 if not USE_LORA else 3e-4
SPARSITY        = 0.20
LORA_RANK       = 8
LORA_ALPHA      = 16
LORA_TARGETS    = ["q", "v"]
ARTIFACTS_DIR   = "artifacts-flan-pruning"
METRICS_CSV     = "pruning_metrics_sheet4.csv"
FINETUNED_PATH  = os.path.join(ARTIFACTS_DIR, f"{short_name}_finetuned.pt")

os.makedirs(ARTIFACTS_DIR, exist_ok=True)

print(f"Model    : {MODEL_NAME}")
print(f"Batch    : {BATCH_SIZE} x {GRAD_ACCUM} accum = effective {BATCH_SIZE*GRAD_ACCUM}")
print(f"Seq len  : input={MAX_INPUT_LEN}, target={MAX_TARGET_LEN}")
print(f"Train    : {BASE_TRAIN_SIZE} samples | Recovery: {RECOVERY_SIZE} | Calib: {CALIB_SIZE}")
print(f"Sparsity : {SPARSITY:.0%}")
print(f"LoRA     : {'ON (rank='+str(LORA_RANK)+')' if USE_LORA else 'OFF — full fine-tune'}")


## 4 · Data — AG News via HuggingFace (`sh0416/ag_news`)

In [ ]:

print("Loading ag_news from HuggingFace …")
hf_data = load_dataset("sh0416/ag_news")
print(hf_data)
print("\nSample:", hf_data["train"][0])


In [ ]:
LABEL_MAP = {1: "world", 2: "sports", 3: "business", 4: "sci/tech"}

def hf_to_seq2seq(split_data):
    rows = []
    for item in split_data:
        snippet = (str(item["title"]) + " - " + str(item["description"]))[:300]
        prompt  = (
            "Classify the following news article into one of four categories: "
            "world, sports, business, or sci/tech. "
            f"Article: '{snippet}'. Predict category:"
        )
        rows.append({
            "input_text":  prompt,
            "target_text": LABEL_MAP[item["label"]],
        })
    return pd.DataFrame(rows)

print("Formatting train split …")
full_train_df = hf_to_seq2seq(hf_data["train"])

print("Formatting test split …")
test_df = hf_to_seq2seq(hf_data["test"])

train_df, val_df = train_test_split(
    full_train_df, test_size=0.10,
    stratify=full_train_df["target_text"], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"\nTrain: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")
print("Label distribution (train):")
print(train_df["target_text"].value_counts())


## 5 · Dataset class & DataLoaders

In [ ]:
class AGNewsSeq2SeqDataset(Dataset):
    def __init__(self, dataframe, tokenizer,
                 max_input=MAX_INPUT_LEN, max_target=MAX_TARGET_LEN):
        self.data       = dataframe
        self.tokenizer  = tokenizer
        self.max_input  = max_input
        self.max_target = max_target

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        enc = self.tokenizer(
            row["input_text"],
            max_length=self.max_input, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        dec = self.tokenizer(
            row["target_text"],
            max_length=self.max_target, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        label_ids = dec.input_ids.squeeze()
        label_ids[label_ids == self.tokenizer.pad_token_id] = -100
        return {
            "input_ids":      enc.input_ids.squeeze(),
            "attention_mask": enc.attention_mask.squeeze(),
            "labels":         label_ids,
        }

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded — vocab size: {tokenizer.vocab_size:,}")


In [ ]:
calib_df      = train_df.iloc[:CALIB_SIZE].reset_index(drop=True)
recovery_df   = train_df.sample(RECOVERY_SIZE,    random_state=42).reset_index(drop=True)
base_train_df = train_df.sample(BASE_TRAIN_SIZE,  random_state=42).reset_index(drop=True)

calib_loader    = DataLoader(AGNewsSeq2SeqDataset(calib_df,      tokenizer), batch_size=BATCH_SIZE, shuffle=False)
recovery_loader = DataLoader(AGNewsSeq2SeqDataset(recovery_df,   tokenizer), batch_size=BATCH_SIZE, shuffle=True)
train_loader    = DataLoader(AGNewsSeq2SeqDataset(base_train_df, tokenizer), batch_size=BATCH_SIZE, shuffle=True)
test_loader     = DataLoader(AGNewsSeq2SeqDataset(test_df,       tokenizer), batch_size=TEST_BATCH)

print(f"Calib    : {len(calib_loader)} batches  ({CALIB_SIZE} samples)")
print(f"Recovery : {len(recovery_loader)} batches  ({RECOVERY_SIZE} samples)")
print(f"Train    : {len(train_loader)} batches  ({BASE_TRAIN_SIZE} samples)")
print(f"Test     : {len(test_loader)} batches  ({len(test_df)} samples)")


## 6 · Helper utilities

In [ ]:
def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def model_size_mb(model, path=None):
    if path and os.path.exists(path):
        return os.path.getsize(path) / (1024 ** 2)
    return sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 ** 2)

def get_prunable_params(model):
    return [(n, p) for n, p in model.named_parameters()
            if "weight" in n and p.dim() > 1]


## 7 · Training loop

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def train_one_epoch(model, loader, optimizer, device, epoch_idx=0):
    model.train()
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

    optimizer.zero_grad()
    for i, batch in enumerate(loader):
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lbls = batch["labels"].to(device)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            loss = model(input_ids=ids, attention_mask=mask, labels=lbls).loss / GRAD_ACCUM

        loss.backward()

        if (i + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                (p for p in model.parameters() if p.requires_grad), CLIP_NORM
            )
            optimizer.step()
            optimizer.zero_grad()

        if i % 25 == 0:
            mem = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0
            print(f"  [epoch {epoch_idx+1}] batch {i:4d} | loss {loss.item()*GRAD_ACCUM:.4f} | GPU {mem:.2f}GB")

    if hasattr(model, "gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()


def wrap_lora(model):
    from peft import LoraConfig, get_peft_model, TaskType

    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGETS,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM,
    )
    model = get_peft_model(model, lora_config)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  [LoRA] trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    return model


def unwrap_lora(model):
    if hasattr(model, "merge_and_unload"):
        model = model.merge_and_unload()
    return model


## 8 · Evaluation

In [ ]:
def evaluate(model, loader, tokenizer, device, save_path=None):
    model.eval()
    label_map  = {"world": 0, "sports": 1, "business": 2, "sci/tech": 3}
    preds, gts = [], []
    total_time = 0.0

    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            lbls = batch["labels"]

            t0  = time.time()
            out = model.generate(
                input_ids=ids,
                attention_mask=mask,
                max_new_tokens=5,
                pad_token_id=tokenizer.pad_token_id,
            )
            total_time += time.time() - t0

            for j in range(ids.size(0)):
                gen   = tokenizer.decode(out[j], skip_special_tokens=True).strip().lower()
                valid = lbls[j][lbls[j] != -100]
                gt    = tokenizer.decode(valid, skip_special_tokens=True).strip().lower()
                preds.append(gen)
                gts.append(gt)

    n      = len(preds)
    y_true = [label_map.get(g, 0) for g in gts]
    y_pred = [label_map.get(p, 0) for p in preds]
    return {
        "Accuracy"  : round(accuracy_score(y_true, y_pred) * 100, 2),
        "Macro-F1"  : round(f1_score(y_true, y_pred, average="macro") * 100, 2),
        "Latency"   : round(total_time / n, 4) if n else 0,
        "Size (MB)" : round(model_size_mb(model, save_path), 2),
    }


## 8.5 · Base Fine-Tune

Fine-tune the raw pretrained model on AG News **before** any pruning.  
This is the key change for 90%+ post-pruning accuracy: prune from a strong base, not random pretrain weights.

In [ ]:
def run_base_finetune(model_name=MODEL_NAME, epochs=FINETUNE_EPOCHS, lr=LR,
                      save_path=FINETUNED_PATH, force_retrain=False):
    if os.path.exists(save_path) and not force_retrain:
        print(f"  [Base FT] Found checkpoint at {save_path} — loading instead of retraining.")
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name, torch_dtype=torch.bfloat16,
            device_map="cuda" if torch.cuda.is_available() else "auto",
        )
        model.load_state_dict(torch.load(save_path, map_location=DEVICE))
        model.eval()
        return model

    print(f"  [Base FT] Fine-tuning {model_name} for {epochs} epoch(s) "
          f"({'LoRA' if USE_LORA else 'full'}) …")
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name, torch_dtype=torch.bfloat16,
        device_map="cuda" if torch.cuda.is_available() else "auto",
    )

    if USE_LORA:
        model = wrap_lora(model)
        trainable_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable_params, lr=lr)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    for ep in range(epochs):
        print(f"  Epoch {ep+1}/{epochs}")
        train_one_epoch(model, train_loader, optimizer, DEVICE, epoch_idx=ep)

    if USE_LORA:
        model = unwrap_lora(model)

    torch.save(model.state_dict(), save_path)
    print(f"  [Base FT] Saved fine-tuned checkpoint → {save_path}")
    model.eval()
    return model

print("Running base fine-tune …")
base_model = run_base_finetune()

print("\nEvaluating fine-tuned base model …")
base_metrics = evaluate(base_model, test_loader, tokenizer, DEVICE, FINETUNED_PATH)
base_metrics["Method"]     = "FINETUNED_BASE"
base_metrics["Model Name"] = MODEL_NAME.split("/")[-1]
print(f"  → Acc {base_metrics['Accuracy']:.2f}%  |  F1 {base_metrics['Macro-F1']:.2f}%  |  "
      f"Latency {base_metrics['Latency']:.4f}s  |  Size {base_metrics['Size (MB)']:.1f} MB")

del base_model
clear_gpu()


## 9 · Pruning Methods

### 9a · Magnitude Pruning

In [ ]:
def magnitude_prune(model, sparsity):
    print(f"  [Magnitude] sparsity={sparsity:.0%}")
    with torch.no_grad():
        for name, param in get_prunable_params(model):
            flat      = param.data.abs().view(-1)
            k         = int(sparsity * flat.numel())
            if k == 0:
                continue
            threshold = torch.kthvalue(flat, k).values
            param.data.mul_(param.data.abs() > threshold)
    return model


### 9b · Movement Pruning

One-shot gradient-based pruning. Uses gradient checkpointing so the single backward pass fits even for large/xl on a T4.

In [ ]:
def movement_prune(model, calib_loader, sparsity, device):
    print(f"  [Movement] collecting gradients (checkpointed) …")
    model.train()
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
    model.zero_grad()

    for batch in calib_loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lbls = batch["labels"].to(device)
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            loss = model(input_ids=ids, attention_mask=mask, labels=lbls).loss
        loss.backward()
        clear_gpu()

    if hasattr(model, "gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()

    print(f"  [Movement] applying mask at sparsity={sparsity:.0%}")
    with torch.no_grad():
        for name, param in get_prunable_params(model):
            if param.grad is None:
                continue
            scores    = (param.data * param.grad).abs()
            k         = int(sparsity * scores.numel())
            if k == 0:
                continue
            threshold = torch.kthvalue(scores.view(-1), k).values
            param.data.mul_(scores > threshold)
            param.grad = None   # free grad memory immediately per-tensor
    model.zero_grad()
    clear_gpu()
    return model


### 9c · SparseGPT

Hessian-compensated column-blocked pruning via forward hooks. Hessians kept on CPU during calibration, moved to GPU one layer at a time during pruning to minimize peak VRAM.

In [ ]:
class SparseGPTLayer:
    def __init__(self, layer):
        self.layer     = layer
        self.n_cols    = layer.weight.shape[1]
        self.H         = torch.zeros(self.n_cols, self.n_cols, dtype=torch.float32)
        self.n_samples = 0

    def add_batch(self, inp):
        x = inp.reshape(-1, self.n_cols).float().cpu()
        self.H        += x.T @ x
        self.n_samples += x.size(0)

    def prune(self, sparsity, device, block_size=128):
        if self.n_samples == 0:
            return
        W = self.layer.weight.data.clone().float()
        H = (self.H / self.n_samples).to(device)
        H.diagonal().add_(0.01 * H.diagonal().mean())   # damping

        try:
            H_inv = torch.cholesky_inverse(torch.linalg.cholesky(H))
        except torch.linalg.LinAlgError:
            H_inv = torch.diag(1.0 / H.diagonal().clamp(min=1e-8))

        mask = torch.zeros_like(W, dtype=torch.bool)
        for start in range(0, self.n_cols, block_size):
            end    = min(start + block_size, self.n_cols)
            W_blk  = W[:, start:end].clone()
            H_blk  = H_inv[start:end, start:end]
            h_diag = H_blk.diagonal().clamp(min=1e-8)

            n_prune = int(sparsity * (end - start))
            if n_prune == 0:
                continue
            scores = W_blk ** 2 / h_diag.unsqueeze(0)
            thresh = torch.kthvalue(scores.reshape(-1), n_prune).values
            blk_mask = scores <= thresh
            mask[:, start:end] = blk_mask
            err = (W_blk * blk_mask.float()) / h_diag.unsqueeze(0)
            W[:, start:end] -= err @ H_blk

        W[mask] = 0.0
        self.layer.weight.data = W.to(self.layer.weight.dtype)
        del H, H_inv, W, mask


def sparsegpt_prune(model, calib_loader, sparsity, device):
    print(f"  [SparseGPT] registering hooks …")
    sg_layers, hooks = {}, {}

    def make_hook(sg):
        def hook(module, inp, out):
            sg.add_batch(inp[0].detach())
        return hook

    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            sg = SparseGPTLayer(module)
            sg_layers[name] = sg
            hooks[name]     = module.register_forward_hook(make_hook(sg))

    print(f"  [SparseGPT] {len(sg_layers)} Linear layers | calibration pass …")
    model.eval()
    with torch.no_grad():
        for batch in calib_loader:
            model(input_ids=batch["input_ids"].to(device),
                  attention_mask=batch["attention_mask"].to(device),
                  labels=batch["labels"].to(device))
            clear_gpu()

    for h in hooks.values():
        h.remove()

    print(f"  [SparseGPT] pruning at sparsity={sparsity:.0%} (layer-by-layer) …")
    for name, sg in sg_layers.items():
        sg.prune(sparsity, device)
        clear_gpu()

    return model


### 9d · HAWQ → bitsandbytes INT8

`bitsandbytes` LLM.int8() — loads directly in INT8, never materializes full-precision weights on GPU. Works for small through xl on a T4.

In [ ]:
def hawq_bnb_quantize(model_name, device):
    from transformers import BitsAndBytesConfig
    import bitsandbytes  # noqa

    print(f"  [bitsandbytes] loading {model_name} in INT8 …")
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model.eval()
    print(f"  [bitsandbytes] INT8 model ready.")
    return model


### 9e · ZeroQuant → quanto INT8

`quanto` weight-only INT8 quantization — works identically across all model sizes, no backprop needed.

In [ ]:
def zeroquant_quanto_quantize(model, device):
    try:
        from quanto import quantize, freeze, qint8
    except ImportError:
        from optimum.quanto import quantize, freeze, qint8

    print(f"  [quanto] applying INT8 weight quantization …")
    quantize(model, weights=qint8)
    freeze(model)
    model = model.to(device)
    print(f"  [quanto] quantization complete.")
    return model

## 10 · Per-method pipeline

In [ ]:
METHOD_REGISTRY = {
    "magnitude"  : {"fn": magnitude_prune,          "type": "prune",        "needs_calib": False},
    "movement"   : {"fn": movement_prune,            "type": "prune",        "needs_calib": True},
    "sparsegpt"  : {"fn": sparsegpt_prune,           "type": "prune",        "needs_calib": True},
    "hawq"       : {"fn": hawq_bnb_quantize,         "type": "quant_bnb",    "needs_calib": False},
    "zeroquant"  : {"fn": zeroquant_quanto_quantize, "type": "quant_quanto", "needs_calib": False},
}


def run_method(method_name, model_name=MODEL_NAME):
    print(f"\n{chr(61)*60}")
    print(f" Method : {method_name.upper()}")
    print(f" Model  : {model_name}  |  Sparsity: {SPARSITY:.0%}  |  LoRA: {USE_LORA}")
    print(f"{chr(61)*60}")

    clear_gpu()
    mem_free = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()
    print(f"  GPU free before load: {mem_free/1024**3:.2f} GB")

    entry       = METHOD_REGISTRY[method_name]
    fn          = entry["fn"]
    mtype       = entry["type"]
    needs_calib = entry["needs_calib"]

    if mtype == "quant_bnb":
        model = fn(model_name, DEVICE)

    elif mtype == "quant_quanto":
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name, torch_dtype=torch.bfloat16,
            device_map="cuda" if torch.cuda.is_available() else "auto",
        )
        if os.path.exists(FINETUNED_PATH):
            model.load_state_dict(torch.load(FINETUNED_PATH, map_location=DEVICE))
            print(f"  Loaded fine-tuned weights.")
        model = fn(model, DEVICE)

    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name, torch_dtype=torch.bfloat16,
            device_map="cuda" if torch.cuda.is_available() else "auto",
        )
        if os.path.exists(FINETUNED_PATH):
            model.load_state_dict(torch.load(FINETUNED_PATH, map_location=DEVICE))
            print(f"  Loaded fine-tuned weights.")
        else:
            print(f"   No checkpoint — pruning raw pretrained weights.")

        if needs_calib:
            model = fn(model, calib_loader, SPARSITY, DEVICE)
        else:
            model = fn(model, SPARSITY)
        clear_gpu()

        if USE_LORA:
            print(f"  Recovery (LoRA r={LORA_RANK}): {RECOVERY_EPOCHS} epoch(s) / "
                  f"{RECOVERY_SIZE} samples  LR={LR_RECOVERY}")
            model = wrap_lora(model)
            trainable_params = [p for p in model.parameters() if p.requires_grad]
            optimizer = torch.optim.AdamW(trainable_params, lr=LR_RECOVERY)
            for ep in range(RECOVERY_EPOCHS):
                train_one_epoch(model, recovery_loader, optimizer, DEVICE, epoch_idx=ep)
            model = unwrap_lora(model)
        else:
            print(f"  Recovery (full FT): {RECOVERY_EPOCHS} epoch(s) / "
                  f"{RECOVERY_SIZE} samples  LR={LR_RECOVERY}")
            optimizer = torch.optim.AdamW(model.parameters(), lr=LR_RECOVERY)
            for ep in range(RECOVERY_EPOCHS):
                train_one_epoch(model, recovery_loader, optimizer, DEVICE, epoch_idx=ep)

    clear_gpu()

    # ── Save ──
    short = model_name.split("/")[-1]
    path  = os.path.join(ARTIFACTS_DIR, f"{short}_{method_name}.pt")
    try:
        torch.save(model.state_dict(), path)
        print(f"  Saved → {path}")
    except Exception as e:
        print(f"  ⚠ Save skipped ({e})")
        path = None

    # ── Evaluate ──
    print("  Evaluating …")
    metrics = evaluate(model, test_loader, tokenizer, DEVICE, path)
    metrics["Method"]     = method_name.upper()
    metrics["Model Name"] = short

    print(f"  → Acc {metrics['Accuracy']:.2f}%  |  "
          f"F1 {metrics['Macro-F1']:.2f}%  |  "
          f"Latency {metrics['Latency']:.4f}s  |  "
          f"Size {metrics['Size (MB)']:.1f} MB")

    del model
    clear_gpu()
    return metrics


## 11 · Run all methods

In [ ]:
METHODS_TO_RUN = ["magnitude", "movement", "sparsegpt", "hawq", "zeroquant"]

all_metrics = []
for method in METHODS_TO_RUN:
    result = run_method(method, MODEL_NAME)
    if result:
        all_metrics.append(result)

print("\nAll methods complete.")


## 12 · Results — Sheet4 format

In [ ]:
if all_metrics:
    # Prepend baseline row
    all_rows = [base_metrics] + all_metrics
    df = pd.DataFrame(all_rows)[["Method", "Model Name", "Accuracy", "Macro-F1", "Latency", "Size (MB)"]]
    display(df)
    df.to_csv(os.path.join(ARTIFACTS_DIR, METRICS_CSV), index=False)
    print(f"\nSaved → {os.path.join(ARTIFACTS_DIR, METRICS_CSV)}")


In [ ]:
ALL_MODELS    = [
    "google/flan-t5-base",
    "google/flan-t5-large",
    "google/flan-t5-xl",
]
METHODS_MULTI = ["magnitude", "movement", "sparsegpt", "hawq", "zeroquant"]

multi_metrics = []

for mname in ALL_MODELS:
    print(f"\n{chr(35)*70}")
    print(f"  SWITCHING MODEL → {mname}")
    print(f"{chr(35)*70}")

    # ── Re-derive T4 profile for this model size ──
    MODEL_NAME = mname
    short_name = mname.split("/")[-1]
    profile    = T4_PROFILES.get(short_name, T4_PROFILES["flan-t5-base"])

    BATCH_SIZE      = profile["batch"]
    GRAD_ACCUM      = profile["grad_accum"]
    MAX_INPUT_LEN   = profile["max_in"]
    MAX_TARGET_LEN  = profile["max_out"]
    CALIB_SIZE      = profile["calib"]
    BASE_TRAIN_SIZE = profile["train_n"]
    RECOVERY_SIZE   = profile["recovery_n"]
    TEST_BATCH      = profile["test_batch"]
    USE_LORA        = profile["use_lora"]
    LR_RECOVERY     = 1e-4 if not USE_LORA else 3e-4
    FINETUNED_PATH  = os.path.join(ARTIFACTS_DIR, f"{short_name}_finetuned.pt")

    print(f"Profile: batch={BATCH_SIZE} accum={GRAD_ACCUM} seq={MAX_INPUT_LEN} "
          f"lora={USE_LORA} train_n={BASE_TRAIN_SIZE}")

    # ── Rebuild tokenizer + loaders for this model ──
    tokenizer = AutoTokenizer.from_pretrained(mname)

    calib_df_m      = train_df.iloc[:CALIB_SIZE].reset_index(drop=True)
    recovery_df_m   = train_df.sample(RECOVERY_SIZE,   random_state=42).reset_index(drop=True)
    base_train_df_m = train_df.sample(BASE_TRAIN_SIZE, random_state=42).reset_index(drop=True)

    calib_loader    = DataLoader(AGNewsSeq2SeqDataset(calib_df_m,      tokenizer), batch_size=BATCH_SIZE, shuffle=False)
    recovery_loader = DataLoader(AGNewsSeq2SeqDataset(recovery_df_m,   tokenizer), batch_size=BATCH_SIZE, shuffle=True)
    train_loader    = DataLoader(AGNewsSeq2SeqDataset(base_train_df_m, tokenizer), batch_size=BATCH_SIZE, shuffle=True)
    test_loader     = DataLoader(AGNewsSeq2SeqDataset(test_df,         tokenizer), batch_size=TEST_BATCH)

    # ── Base fine-tune (skipped if checkpoint already exists) ──
    base_model = run_base_finetune(model_name=mname, save_path=FINETUNED_PATH)
    base_m = evaluate(base_model, test_loader, tokenizer, DEVICE, FINETUNED_PATH)
    base_m["Method"], base_m["Model Name"] = "FINETUNED_BASE", short_name
    multi_metrics.append(base_m)
    del base_model
    clear_gpu()

    # ── Run all methods for this model ──
    for method in METHODS_MULTI:
        res = run_method(method, mname)
        if res:
            multi_metrics.append(res)

print("\n All models complete.")


## 14 · Combined results — all models, all methods

In [ ]:
if multi_metrics:
    mdf = pd.DataFrame(multi_metrics)[[
        "Method", "Model Name", "Accuracy", "Macro-F1", "Latency", "Size (MB)"
    ]]
    display(mdf)
    mdf.to_csv(os.path.join(ARTIFACTS_DIR, "pruning_all_models.csv"), index=False)
    print(f"\nSaved → {os.path.join(ARTIFACTS_DIR, 'pruning_all_models.csv')}")
